# 05. Шаблон прогона одной модели

Один запуск ноутбука = одна модель, один режим (zero-shot или few-shot), один формат ответа (с JSON-схемой или без), один набор карточек.
Результат сохраняется в `results/`: строка на карточку и одна строка в `summary.csv`. Так же ноутбук запускается для всех моделей.

Как запускать (Colab или Kaggle, карта T4):
1. Загрузить в папку `data/` пять файлов: `dns_cards_500.json`, `gold_pairs.csv`, `split_ids.json`, `fewshot.jsonl`, `heldout.jsonl`. В Colab папка лежит на Google Drive.
2. В ячейке «Настройки» выбрать `MODEL`, `MODE`, `SCHEMA`, `SPLIT`.
3. Сначала запустить с `N = 5` (проверка, что всё работает), потом поставить `N = None`.
4. Если запуск прервался, просто запустить ещё раз: готовые карточки пропускаются.

`real_test` не трогаем до этапа 09.

In [ ]:
# Colab / Kaggle: поставить библиотеки (один раз за сессию)
!pip -q install -U transformers accelerate bitsandbytes xgrammar

## Настройки
Модели зафиксированы для сравнения. Claude среди них нет: он только генерирует синтетику.
Для Llama и Gemma на странице модели нужно принять условия, а в Colab добавить секрет `HF_TOKEN`.

In [ ]:
import json, re, time, random, os
import numpy as np
import pandas as pd

MODELS = {
    "qwen35_9b":       "Qwen/Qwen3.5-9B",                          # Alibaba, 9B, Apache 2.0
    "gemma4_e4b":      "google/gemma-4-E4B-it",                    # Google, 4B эффективных параметров, Apache 2.0
    "llama31_8b":      "meta-llama/Llama-3.1-8B-Instruct",         # Meta, 8B, лицензия Llama, нужен доступ
    "ministral3_8b":   "mistralai/Ministral-3-8B-Instruct-2512",   # Mistral AI, 8B, Apache 2.0
    "phi4_mini":       "microsoft/Phi-4-mini-instruct",            # Microsoft, 3.8B, MIT
    "gigachat31_10b":  "ai-sage/GigaChat3.1-10B-A1.8B-bf16",       # Сбер, MoE 10B (1.8B активных), русская, MIT
    "tlite21_8b":      "t-tech/T-lite-it-2.1",                     # Т-Банк, 8B, русская, Apache 2.0
    "yandex5_lite_8b": "yandex/YandexGPT-5-Lite-8B-instruct",      # Яндекс, 8B, русская, своя лицензия
}

MODEL = "qwen35_9b"     # ключ из MODELS
MODE = "few-shot"       # "zero-shot" или "few-shot"
SCHEMA = True           # True: ответ ограничен JSON-схемой, False: модель пишет свободно
SPLIT = "real_val"      # "pilot", "real_val", "heldout" (real_test только на этапе 09)
N = 5                   # сколько карточек взять; None = все
K_SHOT = 5              # сколько примеров в few-shot

DATA = "data"           # папка с входными файлами
OUT = "results"         # сюда пишем результаты
ATTRS = ["диагональ", "разрешение", "частота", "тип матрицы", "яркость", "контрастность", "изогнутость"]

try:                    # в Colab данные и результаты лежат на Google Drive
    from google.colab import drive
    drive.mount("/content/drive")
    DATA, OUT = "/content/drive/MyDrive/vkr1_data", "/content/drive/MyDrive/vkr1_results"
except ImportError:
    pass
os.makedirs(OUT, exist_ok=True)

In [ ]:
# Проверка: все ли имена моделей существуют на Hugging Face (запускать один раз)
from huggingface_hub import HfApi
for k, v in MODELS.items():
    try:
        info = HfApi().model_info(v)
        print("ок  ", k, v, "| доступ по заявке" if info.gated else "")
    except Exception:
        print("НЕТ ", k, v)

## Данные
Карточка = описание и эталон по семи атрибутам (`None`, если в тексте значения нет).
Для реальных карточек эталон берётся из `gold_pairs.csv`, для `heldout` он записан в самом файле.

In [ ]:
cards = {c["url"]: c["description"] for c in json.load(open(f"{DATA}/dns_cards_500.json", encoding="utf-8"))}
pairs = pd.read_csv(f"{DATA}/gold_pairs.csv", dtype=str)
gold_real = {}
for url, g in pairs.groupby("url"):
    gold_real[url] = {a: None for a in ATTRS}          # если атрибута нет в таблице, эталон null
    for _, r in g.iterrows():
        if pd.notna(r["эталон"]):
            gold_real[url][r["атрибут"]] = r["эталон"]
split = json.load(open(f"{DATA}/split_ids.json"))
fewshot = [json.loads(l) for l in open(f"{DATA}/fewshot.jsonl", encoding="utf-8")]
shots = random.Random(0).sample(fewshot, K_SHOT)      # одни и те же примеры для всех моделей

if SPLIT == "heldout":
    syn = [json.loads(l) for l in open(f"{DATA}/heldout.jsonl", encoding="utf-8")]
    items = [{"id": f"heldout{i}", "text": r["description"], "gold": r["gold"]} for i, r in enumerate(syn)]
else:
    items = [{"id": u, "text": cards[u], "gold": gold_real[u]} for u in split[SPLIT]]
if N:
    items = items[:N]
print(SPLIT, "карточек:", len(items))

## Промпт и схема ответа
Промпт одинаковый для всех моделей: одно сообщение пользователя, без system-роли, потому что шаблоны моделей с ней работают по-разному.
В few-shot к инструкции добавляются примеры из `fewshot.jsonl`.

In [ ]:
INSTRUCTION = """Извлеки из описания монитора значения семи характеристик и верни JSON.

Правила:
- Бери только то, что прямо написано в тексте описания. Если характеристики в тексте нет, пиши null.
- Не угадывай и не используй свои знания о модели монитора.
- Все значения строки в таком формате: диагональ — число в дюймах ("27"); разрешение — "2560x1440"; частота — число в герцах ("165"); тип матрицы — "IPS", "VA", "TN" или "OLED"; яркость — число в кд/м² ("300"); контрастность — число ("3000", а 1 млн к 1 пиши "1000000"); изогнутость — "да" или "нет".
- Ответ только JSON с ключами: диагональ, разрешение, частота, тип матрицы, яркость, контрастность, изогнутость."""

def make_prompt(text):
    p = INSTRUCTION
    if MODE == "few-shot":
        for i, ex in enumerate(shots, 1):
            p += f"\n\nПример {i}.\nОписание: {ex['description']}\nОтвет: {json.dumps(ex['gold'], ensure_ascii=False)}"
    return p + f"\n\nОписание: {text}\nОтвет:"

SCHEMA_JSON = {"type": "object",
               "properties": {a: {"type": ["string", "null"]} for a in ATTRS},
               "required": ATTRS, "additionalProperties": False}

print(make_prompt(items[0]["text"])[-700:])

## Загрузка модели
4-битная загрузка (nf4). На T4 нет bf16, поэтому вычисления в fp16. Размышление (thinking) у моделей, где оно есть, выключено: одинаковые условия и меньше токенов.
Если модель выдаёт мусор или `nan`, смотри раздел «Если что-то не работает» в README.

In [ ]:
# GPU
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

name = MODELS[MODEL]
tok = AutoTokenizer.from_pretrained(name)
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
try:
    model = AutoModelForCausalLM.from_pretrained(name, quantization_config=bnb, device_map="auto")
except ValueError:                      # мультимодальные модели иногда грузятся другим классом
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(name, quantization_config=bnb, device_map="auto")
model.eval()
print(name, "| память после загрузки, ГБ:", round(torch.cuda.memory_allocated() / 1e9, 1))

In [ ]:
# GPU
if SCHEMA:
    import xgrammar as xgr
    from xgrammar.contrib.hf import LogitsProcessor
    tok_info = xgr.TokenizerInfo.from_huggingface(tok, vocab_size=model.get_output_embeddings().out_features)
    compiled = xgr.GrammarCompiler(tok_info).compile_json_schema(json.dumps(SCHEMA_JSON), any_whitespace=False)   # один раз

def generate(prompt):
    """возвращает (ответ модели, секунд, сколько токенов сгенерировано)"""
    chat = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    x = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    processors = [LogitsProcessor(compiled)] if SCHEMA else []     # для каждого запроса новый
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**x, max_new_tokens=200, do_sample=False, logits_processor=processors)
    sec = time.time() - t0
    new = out[0][x["input_ids"].shape[1]:]
    return tok.decode(new, skip_special_tokens=True).strip(), sec, len(new)

## Прогон
Карточки идут по одной (batch 1), чтобы честно мерить задержку. Каждая готовая карточка сразу дописывается в файл.

In [ ]:
def parse(raw):
    """словарь по семи атрибутам или None, если ответ не разобрался как JSON"""
    m = re.search(r"\{.*\}", raw, re.S)
    try:
        d = json.loads(m.group()) if m else None
    except json.JSONDecodeError:
        d = None
    if not isinstance(d, dict):
        return None
    return {a: (None if d.get(a) in (None, "", "null") else str(d[a]).strip()) for a in ATTRS}

TAG = f"{SPLIT}__{MODEL}__{MODE}__{'schema' if SCHEMA else 'free'}"
path = f"{OUT}/{TAG}.jsonl"
rows = [json.loads(l) for l in open(path, encoding="utf-8")] if os.path.exists(path) else []
have = {r["id"] for r in rows}

for i, it in enumerate(items):
    if it["id"] in have:
        continue
    raw, sec, ntok = generate(make_prompt(it["text"]))
    row = {"id": it["id"], "gold": it["gold"], "raw": raw, "pred": parse(raw), "sec": round(sec, 2), "tokens": ntok}
    rows.append(row)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
    if i % 10 == 0:
        print(i + 1, "/", len(items), f"| {sec:.1f} с")
print("готово:", path)

## Метрики
Пара «атрибут–значение» считается верной, если значение совпало с эталоном. Если эталон `null`, а модель назвала значение, это ложное значение.
Считаем два варианта: `строго` (текст ответа как есть) и `с нормализацией` (значение приводится правилами из `03_eda`: «27 дюймов» → «27»). Разница между ними показывает, сколько ошибок даёт запись единиц (гипотеза H5).
Доверительные интервалы считаются позже, в ноутбуке сравнения моделей.

In [ ]:
def number(s):
    return re.search(r"\d+(?:[.,]\d+)?", s).group().replace(",", ".")

def resolution(s):
    w, h = re.findall(r"\d+", s)[:2]
    return f"{w}x{h}" if int(w) >= 640 else None

def contrast(s):
    s = re.sub(r"(?<=\d)\s(?=\d{3}\b)", "", s.lower())
    v = float(number(s)) * (10**6 if re.search(r"m|м", s) else 1)
    return str(int(v)) if v >= 100 else None

NORM = {"диагональ": lambda s: f"{float(number(s)):g}", "разрешение": resolution, "частота": number,
        "тип матрицы": lambda s: s.upper().replace("QD-", ""), "яркость": number, "контрастность": contrast,
        "изогнутость": lambda s: s.lower()}

def norm(attr, v):
    try:
        return NORM[attr](v)
    except Exception:
        return v

def f1(tp, fp, fn):
    return 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0

def score(rows, soft):
    cnt = {a: [0, 0, 0] for a in ATTRS}          # tp, fp, fn по атрибутам
    false_null = []                              # для пар с эталоном null: 1, если модель назвала значение
    exact = []                                   # все семь атрибутов верны
    for r in rows:
        pred = r["pred"] or {a: None for a in ATTRS}
        ok = True
        for a in ATTRS:
            g, p = r["gold"][a], pred[a]
            if soft and p is not None:
                p = norm(a, p)
            if g is None:
                false_null.append(p is not None)
                if p is not None:
                    cnt[a][1] += 1; ok = False
            elif p == g:
                cnt[a][0] += 1
            else:
                cnt[a][2] += 1; ok = False
                if p is not None:
                    cnt[a][1] += 1
        exact.append(ok)
    tp, fp, fn = (sum(c[i] for c in cnt.values()) for i in range(3))
    macro = np.mean([f1(*c) for c in cnt.values() if sum(c)])
    return {"micro_f1": f1(tp, fp, fn), "macro_f1": macro, "exact": np.mean(exact),
            "false_null": np.mean(false_null), "per_attr": {a: round(f1(*c), 3) for a, c in cnt.items()}}

strict, soft = score(rows, False), score(rows, True)
sec = [r["sec"] for r in rows]
summary = {"split": SPLIT, "model": MODEL, "mode": MODE, "schema": SCHEMA, "n": len(rows),
           "hf_id": MODELS[MODEL], "k_shot": K_SHOT if MODE == "few-shot" else 0, "дата": time.strftime("%Y-%m-%d %H:%M"),
           "micro_f1_строго": round(strict["micro_f1"], 3), "micro_f1_норм": round(soft["micro_f1"], 3),
           "macro_f1_норм": round(soft["macro_f1"], 3), "exact_норм": round(soft["exact"], 3),
           "ложные_на_null": round(soft["false_null"], 3),
           "невалидных": round(np.mean([r["pred"] is None for r in rows]), 3),
           "сек_среднее": round(np.mean(sec), 2), "сек_p95": round(np.percentile(sec, 95), 2),
           "токенов_среднее": round(np.mean([r["tokens"] for r in rows]), 1)}
try:
    import torch
    summary["память_пик_ГБ"] = round(torch.cuda.max_memory_allocated() / 1e9, 1)
    import transformers
    summary["transformers"] = transformers.__version__
    summary["gpu"] = torch.cuda.get_device_name(0)       # время и память сравнимы только на одной и той же карте
except Exception:
    pass
print(json.dumps(summary, ensure_ascii=False, indent=1))
print("по атрибутам (с нормализацией):", soft["per_attr"])

In [ ]:
# строка в общий файл (старая строка с тем же набором и режимом заменяется)
sp = f"{OUT}/summary.csv"
old = pd.read_csv(sp) if os.path.exists(sp) else pd.DataFrame()
if len(old):
    same = (old["split"] == SPLIT) & (old["model"] == MODEL) & (old["mode"] == MODE) & (old["schema"] == SCHEMA)
    old = old[~same]
pd.concat([old, pd.DataFrame([summary])]).to_csv(sp, index=False)

# первые три карточки: эталон и ответ рядом
for r in rows[:3]:
    print(r["id"][-40:], "\n  эталон:", r["gold"], "\n  ответ: ", r["pred"] if r["pred"] else "НЕ JSON: " + r["raw"][:100])

## Как читать результат
- `micro_f1_норм` — главная метрика. Если `micro_f1_строго` заметно ниже, модель правильно находит значения, но пишет их не в нужном формате.
- `ложные_на_null` — как часто модель придумывает значение, которого в тексте нет. Чем ниже, тем лучше.
- `невалидных` — доля ответов, которые не разобрались как JSON. С JSON-схемой это почти всегда 0.
- `сек_p95` и `память_пик_ГБ` нужны для таблицы сравнения: задержка и сколько памяти заняла модель.